# 🏴 OZZ + RAPTOR v3 — Simplified Pipeline

**DEF CON 34 AI Village HALctf** + **RAPTOR Security Framework**

Pipeline simplificado:
- **RAPTOR**: Análise estática com Semgrep + validação de vulnerabilidades
- **Ozz**: Exploits hardcoded + scan ativo com ferramentas nativas
- **Sem modelo LLM** — sem download de 6GB, roda em <10min

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│   RAPTOR     │────▶│   OZZ        │────▶│  RELATÓRIO   │
│  Semgrep     │     │  Exploits    │     │  Unificado   │
│  Static      │     │  Active      │     │  JSON+HTML   │
└──────────────┘     └──────────────┘     └──────────────┘
```

In [ ]:
# Cell 1: Setup
!pip install -q semgrep requests

import os, sys, time, requests, subprocess, json, re, sqlite3
from pathlib import Path
from datetime import datetime

WORK = '/kaggle/working'
REPORTS = f'{WORK}/reports'
os.makedirs(REPORTS, exist_ok=True)

print('✅ Setup completo!')

In [ ]:
# Cell 2: Clonar repositórios
!git clone --depth 1 https://github.com/Tretabolt/ozz-halctf.git {WORK}/ozz 2>&1 | tail -3
!git clone --depth 1 https://github.com/gadievron/raptor.git {WORK}/raptor 2>&1 | tail -3

OZZ = f'{WORK}/ozz'
RAPTOR = f'{WORK}/raptor'
print(f'✅ Ozz: {OZZ}')
print(f'✅ RAPTOR: {RAPTOR}')

In [ ]:
# Cell 3: 🔬 RAPTOR — Análise estática com Semgrep
print('='*60)
print('🔬 RAPTOR — Static Analysis with Semgrep')
print('='*60)

# Targets do Ozz para análise estática
targets = {
    'target-01-web': f'{OZZ}/universe/target-01',
    'target-03-api': f'{OZZ}/universe/target-03',
    'agent-core': f'{OZZ}/agent',
    'attack-script': f'{OZZ}/attack.py',
}

all_findings = []

for name, path in targets.items():
    if not os.path.exists(path):
        print(f'⚠️ {name}: path not found ({path})')
        continue
    
    print(f'\n🔍 Scanning {name}...')
    sarif_out = f'{REPORTS}/{name}.sarif'
    
    try:
        result = subprocess.run(
            ['semgrep',
             '--config', 'auto',
             '--config', 'p/owasp-top-ten',
             '--config', 'p/security-audit',
             '--sarif',
             '--output', sarif_out,
             '--quiet',
             path],
            capture_output=True, text=True, timeout=120
        )
        
        if os.path.exists(sarif_out):
            with open(sarif_out) as f:
                sarif = json.load(f)
            
            count = 0
            for run in sarif.get('runs', []):
                for r in run.get('results', []):
                    rule_id = r.get('ruleId', 'unknown')
                    msg = r.get('message', {}).get('text', '')
                    locs = r.get('locations', [])
                    fpath = locs[0].get('physicalLocation', {}).get('artifactLocation', {}).get('uri', '') if locs else ''
                    line = locs[0].get('physicalLocation', {}).get('region', {}).get('startLine', 0) if locs else 0
                    
                    # Determine severity
                    sev = 'medium'
                    for run_rule in run.get('tool', {}).get('driver', {}).get('rules', []):
                        if run_rule.get('id') == rule_id:
                            level = run_rule.get('defaultConfiguration', {}).get('level', '')
                            if 'error' in level: sev = 'critical'
                            elif 'warning' in level: sev = 'high'
                            elif 'note' in level: sev = 'low'
                    
                    finding = {
                        'source': 'raptor_semgrep',
                        'target': name,
                        'rule_id': rule_id,
                        'severity': sev,
                        'message': msg,
                        'file': fpath,
                        'line': line
                    }
                    all_findings.append(finding)
                    count += 1
                    print(f'  🚨 [{sev.upper()}] {rule_id}: {msg[:80]}')
            
            print(f'  📊 {count} findings')
        else:
            print(f'  ⚠️ No SARIF output')
            if result.stderr:
                print(f'  stderr: {result.stderr[:200]}')
    except subprocess.TimeoutExpired:
        print(f'  ⏰ Timeout')
    except Exception as e:
        print(f'  ❌ Error: {e}')

with open(f'{REPORTS}/semgrep_findings.json', 'w') as f:
    json.dump(all_findings, f, indent=2)

print(f'\n✅ RAPTOR: {len(all_findings)} findings totais')

In [ ]:
# Cell 4: 🏴 OZZ — Análise do código de ataque
print('='*60)
print('🏴 OZZ — Attack Chain Analysis')
print('='*60)

# Analisar o attack.py do Ozz para extrair vulnerabilidades conhecidas
attack_file = f'{OZZ}/attack.py'
ozz_vulns = []

if os.path.exists(attack_file):
    with open(attack_file) as f:
        code = f.read()
    
    # Extrair endpoints alvo
    urls = re.findall(r'http[s]?://[^\s"\']+', code)
    print(f'\n🎯 Targets encontrados em attack.py:')
    for u in set(urls):
        print(f'  → {u}')
    
    # Extrair técnicas de ataque
    techniques = {
        'SQL Injection': ["admin'--", 'sqli', 'sql_injection', 'OR.*1.*=.*1'],
        'LFI': ['file=/var/', 'lfi', 'local_file_inclusion', '/etc/passwd'],
        'Credential Stuffing': ['sshpass', 'password123', 'admin%password'],
        'SSTI': ['ssti', 'server_side_template', 'jinja2'],
        'JWT Bypass': ['jwt', 'algorithm.*confusion', 'none.*algorithm'],
        'Samba Exploit': ['smbclient', 'CVE-2017-7494', 'samba'],
        'Command Injection': ['command_injection', 'os.system', 'subprocess'],
    }
    
    print(f'\n🔧 Técnicas de ataque detectadas:')
    for tech, patterns in techniques.items():
        found = any(re.search(p, code, re.IGNORECASE) for p in patterns)
        if found:
            ozz_vulns.append({'technique': tech, 'source': 'attack.py analysis'})
            print(f'  ✅ {tech}')
        else:
            print(f'  ❌ {tech}')

# Analisar configuração do universo CTF
docker_compose = f'{OZZ}/universe/docker-compose.yml'
if os.path.exists(docker_compose):
    with open(docker_compose) as f:
        dc = f.read()
    
    print(f'\n🐳 Serviços no Docker Compose:')
    services = re.findall(r'^\s{2}(\w+):', dc, re.MULTILINE)
    for s in services:
        if s not in ('version', 'services', 'networks', 'volumes'):
            print(f'  → {s}')

# Flags esperadas
expected_flags = ['flag{web_master}', 'flag{ssh_ghost}', 'flag{api_breaker}', 'flag{deep_vault}', 'flag{halctf_king}']
print(f'\n🚩 Flags esperadas no CTF:')
for flag in expected_flags:
    print(f'  🚩 {flag}')

print(f'\n✅ OZZ: {len(ozz_vulns)} técnicas de ataque identificadas')

In [ ]:
# Cell 5: 🔗 Cross-reference — RAPTOR findings × Ozz attack techniques
print('='*60)
print('🔗 Cross-Reference: RAPTOR × OZZ')
print('='*60)

# Mapear findings do Semgrep para técnicas do Ozz
mapping = {
    'sql-injection': 'SQL Injection',
    'xss': 'Cross-Site Scripting',
    'path-traversal': 'LFI',
    'hardcoded-password': 'Credential Stuffing',
    'hardcoded-secret': 'Credential Stuffing',
    'jwt': 'JWT Bypass',
    'template-injection': 'SSTI',
    'command-injection': 'Command Injection',
}

with open(f'{REPORTS}/semgrep_findings.json') as f:
    raptor_findings = json.load(f)

correlated = []
for finding in raptor_findings:
    rule = finding['rule_id'].lower()
    matched_technique = None
    for keyword, technique in mapping.items():
        if keyword in rule:
            matched_technique = technique
            break
    
    if matched_technique:
        finding['ozz_technique'] = matched_technique
        finding['correlated'] = True
        correlated.append(finding)
        print(f'  🔗 {finding["rule_id"]} → {matched_technique}')

print(f'\n📊 {len(correlated)}/{len(raptor_findings)} findings correlacionados com técnicas do Ozz')

# Salvar correlação
with open(f'{REPORTS}/correlated_findings.json', 'w') as f:
    json.dump(correlated, f, indent=2)

In [ ]:
# Cell 6: 📊 Relatório Final Unificado
print('='*60)
print('📊 RELATÓRIO UNIFICADO — RAPTOR + OZZ')
print('='*60)

with open(f'{REPORTS}/semgrep_findings.json') as f:
    raptor_findings = json.load(f)

with open(f'{REPORTS}/correlated_findings.json') as f:
    correlated = json.load(f)

# Contar por severidade
severity_count = {'critical': 0, 'high': 0, 'medium': 0, 'low': 0}
for f in raptor_findings:
    sev = f.get('severity', 'medium')
    severity_count[sev] = severity_count.get(sev, 0) + 1

# Gerar relatório
report = {
    'pipeline': 'OZZ + RAPTOR v3',
    'timestamp': datetime.now().isoformat(),
    'raptor': {
        'tool': 'Semgrep (auto + OWASP Top 10 + Security Audit)',
        'total_findings': len(raptor_findings),
        'by_severity': severity_count,
        'findings': raptor_findings
    },
    'ozz': {
        'attack_techniques': len(ozz_vulns),
        'techniques': ozz_vulns,
        'expected_flags': expected_flags,
        'targets': list(set(re.findall(r'http[s]?://[^\s"\']+', open(f'{OZZ}/attack.py').read()))) if os.path.exists(f'{OZZ}/attack.py') else []
    },
    'correlation': {
        'correlated_findings': len(correlated),
        'details': correlated
    }
}

with open(f'{REPORTS}/unified_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Imprimir resumo
print(f"\n{'='*50}")
print(f"  🔬 RAPTOR (Static Analysis)")
print(f"     Total findings: {len(raptor_findings)}")
print(f"     🔴 Critical: {severity_count['critical']}")
print(f"     🟠 High:     {severity_count['high']}")
print(f"     🟡 Medium:   {severity_count['medium']}")
print(f"     🟢 Low:      {severity_count['low']}")
print(f"\n  🏴 OZZ (Attack Analysis)")
print(f"     Técnicas: {len(ozz_vulns)}")
print(f"     Flags esperadas: {len(expected_flags)}")
print(f"\n  🔗 Correlação")
print(f"     Findings correlacionados: {len(correlated)}")
print(f"{'='*50}")
print(f"\n📄 Relatório: {REPORTS}/unified_report.json")

In [ ]:
# Cell 7: 📋 Detalhamento dos findings por target
print('='*60)
print('📋 FINDINGS POR TARGET')
print('='*60)

with open(f'{REPORTS}/semgrep_findings.json') as f:
    findings = json.load(f)

by_target = {}
for f in findings:
    t = f['target']
    if t not in by_target:
        by_target[t] = []
    by_target[t].append(f)

for target, target_findings in by_target.items():
    print(f'\n🎯 {target} ({len(target_findings)} findings)')
    print(f'  {"─"*40}')
    for f in target_findings:
        sev_icon = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢'}.get(f['severity'], '⚪')
        print(f'  {sev_icon} [{f["severity"].upper()}] {f["rule_id"]}')
        print(f'     📄 {f["file"]}:{f["line"]}')
        print(f'     💬 {f["message"][:100]}')
        if f.get('ozz_technique'):
            print(f'     🔗 Ozz technique: {f["ozz_technique"]}')
        print()

In [ ]:
# Cell 8: Relatório HTML
html = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>OZZ + RAPTOR Report</title>
<style>
body {{ font-family: monospace; background: #0a0a0a; color: #00ff00; padding: 20px; }}
h1 {{ color: #00ff00; border-bottom: 2px solid #00ff00; }}
h2 {{ color: #00cc00; }}
.card {{ background: #111; border: 1px solid #333; padding: 15px; margin: 10px 0; border-radius: 5px; }}
.critical {{ color: #ff0000; }}
.high {{ color: #ff8800; }}
.medium {{ color: #ffff00; }}
.low {{ color: #00ff00; }}
.stat {{ font-size: 2em; color: #00ff00; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #333; padding: 8px; text-align: left; }}
th {{ background: #222; }}
</style></head><body>
<h1>🏴 OZZ + RAPTOR — Unified Pentest Report</h1>
<p>Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</p>

<h2>🔬 RAPTOR — Static Analysis</h2>
<div class="card">
<p>Total Findings: <span class="stat">{len(raptor_findings)}</span></p>
<p><span class="critical">🔴 Critical: {severity_count["critical"]}</span> | 
<span class="high">🟠 High: {severity_count["high"]}</span> | 
<span class="medium">🟡 Medium: {severity_count["medium"]}</span> | 
<span class="low">🟢 Low: {severity_count["low"]}</span></p>
</div>

<h2>🏴 OZZ — Attack Techniques</h2>
<div class="card">
<p>Techniques identified: <span class="stat">{len(ozz_vulns)}</span></p>
<ul>{"".join(f"<li>{v["technique"]}</li>" for v in ozz_vulns)}</ul>
</div>

<h2>🔗 Correlation</h2>
<div class="card">
<p>Correlated findings: <span class="stat">{len(correlated)}</span></p>
</div>

<h2>📋 Detailed Findings</h2>
<table>
<tr><th>Severity</th><th>Rule</th><th>Target</th><th>File</th><th>Line</th><th>Description</th></tr>
'''  

for f in raptor_findings:
    sev = f['severity']
    html += f'<tr><td class="{sev}">{sev.upper()}</td><td>{f["rule_id"]}</td><td>{f["target"]}</td><td>{f["file"]}</td><td>{f["line"]}</td><td>{f["message"][:80]}</td></tr>\n'

html += '''</table>
</body></html>'''

with open(f'{REPORTS}/report.html', 'w') as f:
    f.write(html)

print(f'📄 HTML report: {REPORTS}/report.html')
print(f'📄 JSON report: {REPORTS}/unified_report.json')
print(f'\n🏴 Pipeline OZZ + RAPTOR v3 finalizado!')